# 3D Brain Tumor Segmentation - Quick Start Tutorial

This notebook demonstrates how to use the 3D multi-modal brain tumor segmentation pipeline.

## Setup

First, install the required dependencies:

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -r ../requirements.txt
# !pip install -e ..

## 1. Import Libraries

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
from pathlib import Path

from brain_tumor_segmentation.utils import get_device, set_seed
from brain_tumor_segmentation.utils.config import load_config
from brain_tumor_segmentation.models import build_model
from brain_tumor_segmentation.data import simple_transform

print("Imports successful!")

## 2. Setup Device and Seed

In [ ]:
# Auto-detect device (CUDA > MPS > CPU)
device = get_device("auto")
print(f"Using device: {device}")

# Set random seed for reproducibility
set_seed(42)
print("Random seed set to 42")

## 3. Load Configuration

In [ ]:
# Load default configuration
config = load_config("../configs/default_config.yaml")

# Print some key settings
print(f"Model: {config.model.name}")
print(f"Batch size: {config.training.batch_size}")
print(f"Number of epochs: {config.training.num_epochs}")

## 4. Build Model

In [ ]:
# Build 3D U-Net model
model = build_model(
    model_name=config.model.name,
    spatial_dims=config.model.spatial_dims,
    in_channels=config.model.in_channels,
    out_channels=config.model.out_channels,
    channels=[16, 32, 64],  # Smaller for demo
    strides=[2, 2],
    num_res_units=config.model.num_res_units,
    dropout=config.model.dropout,
)

model = model.to(device)

# Count parameters
num_params = sum(p.numel() for p in model.parameters())
print(f"Model has {num_params:,} parameters")

## 5. Test Forward Pass

In [ ]:
# Create dummy input (4 modalities, 64x64x64 volume)
dummy_input = torch.randn(1, 4, 64, 64, 64).to(device)

# Forward pass
model.eval()
with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output range: [{output.min():.2f}, {output.max():.2f}]")

## 6. Get Predictions

In [ ]:
# Convert logits to predictions
probs = torch.softmax(output, dim=1)
predictions = torch.argmax(probs, dim=1)

print(f"Predictions shape: {predictions.shape}")
print(f"Unique classes: {torch.unique(predictions).cpu().numpy()}")

# Class distribution
for class_idx in range(4):
    count = (predictions == class_idx).sum().item()
    percentage = 100 * count / predictions.numel()
    print(f"Class {class_idx}: {count:6d} voxels ({percentage:.2f}%)")

## 7. Training Example

For actual training with the MSD Task01 dataset, use the command-line script:

```bash
python scripts/train.py \
    --config configs/default_config.yaml \
    --data-root ./data/Task01_BrainTumour
```

Or in a notebook/Colab:

In [ ]:
# Training in notebook (uncomment to run)
# !python ../scripts/train.py \
#     --config ../configs/quick_train.yaml \
#     --data-root ../data/Task01_BrainTumour

## 8. Inference Example

After training, run inference:

In [ ]:
# Inference in notebook (uncomment to run)
# !python ../scripts/inference.py \
#     --checkpoint ../checkpoints/best_model.pth \
#     --input ../data/Task01_BrainTumour/imagesTr/BRATS_001.nii.gz \
#     --output prediction.nii.gz \
#     --post-process

## 9. Next Steps

- Download the MSD Task01 dataset
- Train the model on the full dataset
- Evaluate on test set
- Generate explainability visualizations

See the README for more detailed instructions!